In [ ]:
import cv2
import os
import math

In [ ]:
def get_video_info(video_path: str) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {"error": "Cannot open video file"}
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    cap.release()
    return {
        "fps": fps,
        "total_frames": total_frames,
        "duration_seconds": duration,
        "duration_hms": f"{int(duration // 3600):02}:{int((duration % 3600) // 60):02}:{int(duration % 60):02}",
    }

def extract_frames(video_path, output_folder, frames_per_second=1.0):
    """
    Extract frames at a target rate (can be fractional). Picks frames at times:
      t = 0, 1/fps_target, 2/fps_target, ... < duration
    Converts times to frame indices via round(t * fps). Uses seeking for efficiency.
    """
    os.makedirs(output_folder, exist_ok=True)

    info = get_video_info(video_path)
    if "error" in info:
        return info

    fps = info["fps"]
    duration = info["duration_seconds"]
    total_frames = info["total_frames"]
    if fps <= 0 or total_frames == 0 or duration == 0:
        return {"error": "Video has invalid metadata (fps/frames/duration).", **info}

    if frames_per_second <= 0:
        return {"error": "frames_per_second must be > 0.", **info}

    # How many timestamps we’ll sample (include t=0, stop before duration)
    num_samples = math.ceil(duration * frames_per_second)
    # Times in seconds: 0, 1/fps_target, 2/fps_target, ...
    target_times = [(i / frames_per_second) for i in range(num_samples)]
    # Convert to frame indices; clamp to last frame; dedupe + sort
    frame_indices = sorted(
        {min(int(round(t * fps)), total_frames - 1) for t in target_times if t < duration}
    )
    # Edge case: if duration is extremely close to an exact multiple, ensure at least t=0 included
    if 0 not in frame_indices:
        frame_indices = [0] + frame_indices

    cap = cv2.VideoCapture(video_path)
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    extracted_count = 0

    for idx in frame_indices:
        # Seek to the frame; some codecs land on nearest keyframe, so we still read once
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        extracted_count += 1
        # Optional: include timestamp in filename for clarity
        ts_sec = idx / fps
        frame_filename = f"{base_name}_{extracted_count:04d}_t{ts_sec:.2f}s.jpg"
        cv2.imwrite(os.path.join(output_folder, frame_filename), frame)

    cap.release()

    return {
        **info,
        "frames_requested_per_second": frames_per_second,
        "frames_planned": len(frame_indices),
        "frames_extracted": extracted_count,
        "output_folder": output_folder,
        "sampling_rule": "t = 0, 1/fps_target, 2/fps_target, … < duration; index = round(t * fps)",
    }

In [11]:
# Example usage
video_path = '../sample/video_input/Do you notice these signs in your toddler.mp4'
output_folder = '../sample/frames_output'
print(get_video_info(video_path))
extract_frames(video_path, output_folder)

{'fps': 30.0, 'total_frames': 748, 'duration_seconds': 24.933333333333334, 'duration_hms': '00:00:24'}


{'fps': 30.0,
 'total_frames': 748,
 'duration_seconds': 24.933333333333334,
 'duration_hms': '00:00:24',
 'frames_requested_per_second': 1.0,
 'frames_planned': 25,
 'frames_extracted': 25,
 'output_folder': '../sample/frames_output',
 'sampling_rule': 't = 0, 1/fps_target, 2/fps_target, … < duration; index = round(t * fps)'}